In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Small Business Loan Agent - ADK Evaluation

This notebook demonstrates how to evaluate the Small Business Loan Agent using ADK evaluation criteria.

**Agent Overview:**
- 4 sub-agents: DocumentExtraction, Underwriting, Pricing, LoanDecision
- Orchestrator pattern with AgentTool
- Human-in-the-loop approval gate
- Firestore state management for repair & resume

**Prerequisite:** You need to have `google-adk[eval]` installed

## Setup

In [2]:
import json
import os
import sys

from pathlib import Path

# Ensure we run from the agent root so adk eval can find the agent module
os.chdir(Path(__file__).parent.parent if "__file__" in dir() else Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

from dotenv import load_dotenv  # noqa: E402
from eval.eval_utils import (  # noqa: E402
    create_pre_repaired_state_in_firestore,
    generate_random_sbl_id,
    get_mime_type,
    load_file_as_base64,
)

load_dotenv()

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"

# adk eval expects the path to the package directory containing __init__.py and agent.py
AGENT_MODULE_PATH = "small_business_loan_agent"
EVAL_CONFIG_PATH = "eval/eval_config.json"
EVAL_DATA_PATH = "eval"
SAMPLE_APPS_PATH = "data/sample_applications"

print(f"Agent module path: {AGENT_MODULE_PATH}")

Agent module path: small_business_loan_agent


---

## Evaluation Dataset

ADK eval supports multi-turn conversations where the `conversation` array contains multiple invocations (turns).

We will create 4 test cases:
1. **Happy Path with Approval** - Full end-to-end flow (2 turns)
2. **Stop for Reparation** - Incomplete application with missing fields (1 turn)
3. **Resume After Repair** - Resume from UnderwritingAgent after offline repair (1 turn)

In [3]:
eval_cases = []

### Test Case 1: Happy Path with Approval

**Description:** Full end-to-end flow with 2 turns
- **Turn 1:** User submits complete application PDF -> Agent processes through pricing -> Returns approval prompt
- **Turn 2:** User approves ("yes") -> Agent finalizes loan decision

In [4]:
COMPLETE_PDF_PATH = f"{SAMPLE_APPS_PATH}/sample_application_complete.pdf"

if Path(COMPLETE_PDF_PATH).exists():
    pdf_base64 = load_file_as_base64(COMPLETE_PDF_PATH)
    pdf_mime_type = get_mime_type(COMPLETE_PDF_PATH)
    print(f"Loaded PDF: {COMPLETE_PDF_PATH}")
else:
    print(f"ERROR: Sample PDF not found at {COMPLETE_PDF_PATH}")
    pdf_base64 = None
    pdf_mime_type = None


if pdf_base64:
    multi_turn_sbl_id = generate_random_sbl_id()

    expected_response_turn1 = (
        "Loan Application Summary:\n"
        "- Business: Cymbal Coffee Roasters LLC\n"
        "- Owner: Jane Doe\n"
        "- Loan Amount: $150,000\n"
        "- Annual Revenue: $850,000\n"
        "- Eligibility: ELIGIBLE\n"
        "- Risk Tier: Tier 1 - Low Risk\n"
        "- Interest Rate: 6.5%\n"
        "- Monthly Payment: $2,934.92\n"
        "- Total Interest: $26,095.33\n\n"
        "Do you approve this loan? (yes/no)"
    )
    expected_response_turn2 = (
        "Loan processing completed successfully.\n"
        f"- Loan {multi_turn_sbl_id} has been approved.\n"
        "- Approved for $150,000 at 6.5% for 60 months.\n"
        "- Decision letter generated."
    )

    happy_path_approval = {
        "eval_id": "happy_path_with_approval",
        "conversation": [
            {
                "invocation_id": "e-001-analyze",
                "user_content": {
                    "parts": [
                        {"text": f"Process this loan application for {multi_turn_sbl_id}"},
                        {
                            "inline_data": {
                                "mime_type": pdf_mime_type,
                                "data": pdf_base64,
                            }
                        },
                    ],
                    "role": "user",
                },
                "final_response": {
                    "parts": [{"text": expected_response_turn1}],
                    "role": "model",
                },
                "intermediate_data": {
                    "tool_uses": [
                        {"name": "check_process_status", "args": {}},
                        {"name": "DocumentExtractionAgent", "args": {}},
                        {"name": "UnderwritingAgent", "args": {}},
                        {"name": "PricingAgent", "args": {}},
                    ]
                },
            },
            {
                "invocation_id": "e-002-approve",
                "user_content": {"parts": [{"text": "yes"}], "role": "user"},
                "final_response": {
                    "parts": [{"text": expected_response_turn2}],
                    "role": "model",
                },
                "intermediate_data": {"tool_uses": [{"name": "LoanDecisionAgent", "args": {}}]},
            },
        ],
        "session_input": {
            "app_name": "small_business_loan_agent",
            "user_id": "loan_officer_001",
        },
    }
    eval_cases.append(happy_path_approval)

    print("Test Case Added: happy_path_with_approval")
    print(f"   SBL ID: {multi_turn_sbl_id}")
    print("   Turns: 2 (process -> approve)")
else:
    print("Skipped: happy_path_with_approval (PDF not loaded)")

Loaded PDF: data/sample_applications/sample_application_complete.pdf
Test Case Added: happy_path_with_approval
   SBL ID: SBL-2025-52480
   Turns: 2 (process -> approve)


### Test Case 2: Stop for Reparation (Missing Fields)

**Description:** Agent stops processing when critical data is missing
- User submits an incomplete application (missing annual_revenue, net_profit, loan_amount)
- Agent processes through DocumentExtraction
- Agent stops and requests repair before proceeding

In [5]:
INCOMPLETE_PDF_PATH = f"{SAMPLE_APPS_PATH}/sample_application_incomplete.pdf"

if Path(INCOMPLETE_PDF_PATH).exists():
    incomplete_pdf_base64 = load_file_as_base64(INCOMPLETE_PDF_PATH)
    incomplete_pdf_mime_type = get_mime_type(INCOMPLETE_PDF_PATH)
    reparation_sbl_id = generate_random_sbl_id()

    stop_for_reparation_case = {
        "eval_id": "stop_for_reparation_missing_fields",
        "conversation": [
            {
                "invocation_id": "e-001-reparation",
                "user_content": {
                    "parts": [
                        {"text": f"Process this loan application for {reparation_sbl_id}"},
                        {
                            "inline_data": {
                                "mime_type": incomplete_pdf_mime_type,
                                "data": incomplete_pdf_base64,
                            }
                        },
                    ],
                    "role": "user",
                },
                "final_response": {
                    "parts": [
                        {
                            "text": "Missing 1 critical field(s): loan_amount_requested. The workflow has been stopped. Please provide the loan amount requested to proceed."
                        }
                    ],
                    "role": "model",
                },
                "intermediate_data": {
                    "tool_uses": [
                        {"name": "check_process_status", "args": {}},
                        {"name": "DocumentExtractionAgent", "args": {}},
                    ]
                },
            }
        ],
        "session_input": {
            "app_name": "small_business_loan_agent",
            "user_id": "loan_officer_001",
        },
    }
    eval_cases.append(stop_for_reparation_case)

    print("Test Case Added: stop_for_reparation_missing_fields")
    print(f"   SBL ID: {reparation_sbl_id}")
    print("   Turns: 1 (incomplete application)")
else:
    print(f"Skipped: stop_for_reparation_missing_fields (PDF not found at {INCOMPLETE_PDF_PATH})")

Test Case Added: stop_for_reparation_missing_fields
   SBL ID: SBL-2025-04446
   Turns: 1 (incomplete application)


### Test Case 3: Resume After Repair

**Description:** Agent resumes from where it left off after offline repair

**Scenario:**
1. A previous process stopped at DocumentExtractionAgent due to missing fields
2. An admin repaired the data offline (added the missing fields in Firestore)
3. User asks to resume processing
4. Agent should resume from UnderwritingAgent (skipping DocumentExtractionAgent)

In [6]:
resume_sbl_id = generate_random_sbl_id()

# Create pre-repaired state in Firestore
create_pre_repaired_state_in_firestore(resume_sbl_id)

expected_response_resume = (
    "Loan Application Summary:\n"
    "- Business: Cymbal Coffee Roasters LLC\n"
    "- Owner: Jane Doe\n"
    "- Loan Amount: $150,000\n"
    "- Annual Revenue: $850,000\n"
    "- Eligibility: ELIGIBLE\n"
    "- Risk Tier: Tier 1 - Low Risk\n"
    "- Interest Rate: 6.5%\n\n"
    "Do you approve this loan? (yes/no)"
)

resume_after_repair_case = {
    "eval_id": "resume_after_repair",
    "conversation": [
        {
            "invocation_id": "e-001-resume",
            "user_content": {
                "parts": [{"text": f"Resume processing for {resume_sbl_id}"}],
                "role": "user",
            },
            "final_response": {
                "parts": [{"text": expected_response_resume}],
                "role": "model",
            },
            "intermediate_data": {
                "tool_uses": [
                    {"name": "check_process_status", "args": {}},
                    {"name": "UnderwritingAgent", "args": {}},
                    {"name": "PricingAgent", "args": {}},
                ]
            },
        }
    ],
    "session_input": {
        "app_name": "small_business_loan_agent",
        "user_id": "loan_officer_001",
    },
}
eval_cases.append(resume_after_repair_case)

print("Test Case Added: resume_after_repair")
print(f"   SBL ID: {resume_sbl_id}")
print("   Turns: 1 (resume after offline repair)")
print("   Note: Pre-repaired state created in Firestore")

Created pre-repaired state for SBL-2025-15099
Test Case Added: resume_after_repair
   SBL ID: SBL-2025-15099
   Turns: 1 (resume after offline repair)
   Note: Pre-repaired state created in Firestore


---

### Save Evaluation Dataset

In [7]:
eval_set = {
    "eval_set_id": "small_business_loan_agent_eval_set",
    "name": "Small Business Loan Agent Evaluation Set",
    "description": "Includes single-turn and multi-turn test cases for loan processing workflow",
    "eval_cases": eval_cases,
}

eval_set_path = f"{EVAL_DATA_PATH}/small_business_loan_eval_set.evalset.json"
with open(eval_set_path, "w") as f:
    json.dump(eval_set, f, indent=2)

print("\n" + "=" * 60)
print("EVALUATION DATASET SUMMARY")
print("=" * 60)
for i, case in enumerate(eval_cases, 1):
    print(f"  {i}. {case['eval_id']}")
    print(f"     Turns: {len(case['conversation'])}")
print("=" * 60)
print(f"Total test cases: {len(eval_cases)}")
print(f"Saved to: {eval_set_path}")


EVALUATION DATASET SUMMARY
  1. happy_path_with_approval
     Turns: 2
  2. stop_for_reparation_missing_fields
     Turns: 1
  3. resume_after_repair
     Turns: 1
Total test cases: 3
Saved to: eval/small_business_loan_eval_set.evalset.json


---

## Run Evaluation

Run individual test cases or the full eval set using `adk eval`.

In [8]:
!adk eval {AGENT_MODULE_PATH} --config_file_path {EVAL_CONFIG_PATH} {EVAL_DATA_PATH}/small_business_loan_eval_set.evalset.json:happy_path_with_approval --print_detailed_results

/usr/local/google/home/norafilali/projects/adk-samples/python/agents/small-business-loan-agent/.venv/lib/python3.11/site-packages/google/adk/evaluation/metric_evaluator_registry.py:106: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/usr/local/google/home/norafilali/projects/adk-samples/python/agents/small-business-loan-agent/.venv/lib/python3.11/site-packages/google/adk/evaluation/local_eval_service.py:124: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  user_simulator_provider: UserSimulatorProvider = UserSimulatorProvider(),
Using evaluation criteria: criteria={'rubric_based_tool_use_quality_v1': BaseCriterion(threshold=0.8, judge_model_optio

In [9]:
!adk eval {AGENT_MODULE_PATH} --config_file_path {EVAL_CONFIG_PATH} {EVAL_DATA_PATH}/small_business_loan_eval_set.evalset.json:stop_for_reparation_missing_fields --print_detailed_results

/usr/local/google/home/norafilali/projects/adk-samples/python/agents/small-business-loan-agent/.venv/lib/python3.11/site-packages/google/adk/evaluation/metric_evaluator_registry.py:106: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/usr/local/google/home/norafilali/projects/adk-samples/python/agents/small-business-loan-agent/.venv/lib/python3.11/site-packages/google/adk/evaluation/local_eval_service.py:124: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  user_simulator_provider: UserSimulatorProvider = UserSimulatorProvider(),
Using evaluation criteria: criteria={'rubric_based_tool_use_quality_v1': BaseCriterion(threshold=0.8, judge_model_optio

In [10]:
!adk eval {AGENT_MODULE_PATH} --config_file_path {EVAL_CONFIG_PATH} {EVAL_DATA_PATH}/small_business_loan_eval_set.evalset.json:resume_after_repair --print_detailed_results

/usr/local/google/home/norafilali/projects/adk-samples/python/agents/small-business-loan-agent/.venv/lib/python3.11/site-packages/google/adk/evaluation/metric_evaluator_registry.py:106: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/usr/local/google/home/norafilali/projects/adk-samples/python/agents/small-business-loan-agent/.venv/lib/python3.11/site-packages/google/adk/evaluation/local_eval_service.py:124: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  user_simulator_provider: UserSimulatorProvider = UserSimulatorProvider(),
Using evaluation criteria: criteria={'rubric_based_tool_use_quality_v1': BaseCriterion(threshold=0.8, judge_model_optio

2026-03-24 01:24:47,586 - INFO - google_llm.py:250 - Response received from the model.
2026-03-24 01:24:48,870 - INFO - state_service.py:148 - Updated step UnderwritingAgent to status completed for request_id: SBL-2025-00871
2026-03-24 01:24:48,870 - INFO - state_callbacks.py:170 - Marked UnderwritingAgent as completed
2026-03-24 01:24:48,871 - INFO - runners.py:1571 - Closing runner...
2026-03-24 01:24:48,871 - INFO - runners.py:1579 - Runner closed.
2026-03-24 01:24:48,915 - INFO - google_llm.py:185 - Sending out request, model: gemini-3.1-pro-preview, backend: GoogleLLMVariant.VERTEX_AI, stream: False
2026-03-24 01:24:53,581 - INFO - google_llm.py:250 - Response received from the model.
2026-03-24 01:24:54,773 - INFO - plugin_manager.py:104 - Plugin 'request_intercepter_plugin' registered.
2026-03-24 01:24:54,773 - INFO - plugin_manager.py:104 - Plugin 'ensure_retry_options' registered.
2026-03-24 01:24:55,991 - INFO - state_callbacks.py:98 - Loaded process state for PricingAgent
20

---

## Evaluation Criteria

| Criterion | Purpose | Threshold | Reference Response Required |
|-----------|---------|----------|----------------------------|
| `rubric_based_tool_use_quality_v1` | Validate tool ordering using LLM judge | 0.8 | No |
| `rubric_based_final_response_quality_v1` | Evaluate response quality without reference (LLM judge) | 0.8 | No |
| `final_response_match_v2` | Semantic equivalence of response (LLM-based) | 0.7 | Yes |

### Key Design Decisions

1. **`rubric_based_tool_use_quality_v1` instead of `tool_trajectory_avg_score`**: When using `AgentTool` wrapper, the orchestrator LLM dynamically generates `request` args containing context from previous steps. Since these args are LLM-generated, they cannot be predicted exactly. The rubric-based approach uses an LLM judge to validate tool ordering semantically.

2. **`final_response_match_v2` for semantic matching**: More flexible than exact string matching. Uses LLM to evaluate whether the actual response is semantically equivalent to the expected reference response. Threshold set to 0.7 to account for natural LLM wording variation.

3. **`rubric_based_final_response_quality_v1` for reference-free evaluation**: Evaluates the quality of the final response using custom rubrics without requiring a reference response. Useful for assessing completeness, clarity, and actionability.

### Tool Use Rubrics

| Rubric ID | Rule |
|-----------|------|
| `status_first` | `check_process_status` is called before any agent tools on initial request |
| `extraction_before_underwriting` | `DocumentExtractionAgent` is called before `UnderwritingAgent` |
| `underwriting_before_pricing` | `UnderwritingAgent` is called before `PricingAgent` |
| `approval_required` | `LoanDecisionAgent` is only called after user approval |
| `pricing_before_decision` | `PricingAgent` is called before `LoanDecisionAgent` |

### Response Quality Rubrics

| Rubric ID | Rule |
|-----------|------|
| `loan_summary_completeness` | Response includes business name, owner, loan amount, revenue, eligibility, risk tier, rate, payment |
| `clear_next_step` | Response clearly indicates next action (approval prompt, completion, status, or missing info) |
| `error_handling_clarity` | When data is missing, response clearly identifies what is missing |

### Notes

- Document is passed as `inline_data` in user content
- Gemini 3.1 Pro reads the PDF natively
- Uses randomly generated SBL ID to avoid Firestore collisions
- Expected tool sequence: `check_process_status` -> `DocumentExtractionAgent` -> `UnderwritingAgent` -> `PricingAgent`
- Expected response includes: business name, owner, loan amount, revenue, eligibility, risk tier, rate, payment

**Resources:**
- [ADK Evaluation docs](https://google.github.io/adk-docs/evaluate/)
- [Evaluation Criteria](https://google.github.io/adk-docs/evaluate/criteria/)
- [ADK Samples - Evaluation](https://github.com/google/adk-samples/blob/main/python/notebooks/evaluation/evaluation_criteria_in_adk.ipynb)